# Multi-Vendor WFO to Cross-Framework Forward Test

This notebook tests a fixed walk-forward shape across data vendors and backtesting frameworks. Each provider is treated as its own research path: `backtesting.py` optimizes MAG7 portfolio SMA crossover parameters on the in-sample window, then the selected parameters are forward-tested independently on Zipline Reloaded and NautilusTrader.

The second half asks whether a constrained portfolio of multiple parameter/framework/vendor strategies can improve out-of-sample performance versus the single best forward-test row.

Data and SMA features come from Quant Warehouse. The backtesting engines receive prepared in-memory frames and do not compute indicators or read temporary files.

In [1]:
from __future__ import annotations

import json
import subprocess
import sys
from io import StringIO
from pathlib import Path
from time import perf_counter
import warnings

warnings.filterwarnings("ignore", message="Jupyter Notebook detected.*", category=UserWarning)
warnings.filterwarnings("ignore", message="DataFrameGroupBy.apply operated on the grouping columns.*", category=FutureWarning)
warnings.filterwarnings("ignore", message="The behavior of DataFrame concatenation with empty or all-NA entries is deprecated.*", category=FutureWarning)

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

REPO_ROOT = Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from quant_orchestrator.optimization import (
    MetricFilter,
    filter_results,
    optimize_random_mean_variance_weights,
    returns_matrix_from_equity_curves,
)
from quant_orchestrator.pipeline import PipelineContext
from quant_orchestrator.backtests.research import (
    FAST_WINDOWS,
    SLOW_WINDOWS,
    FrameworkRun,
    build_sma_frame,
    run_backtesting_py,
    run_zipline,
)
from quant_orchestrator.experiments import WindowSpec, build_walk_forward_windows
from quant_orchestrator.platforms.backtesting_frameworks.reporting import (
    build_common_summary,
    normalize_equity_curve,
)
from quant_orchestrator.platforms.backtesting_frameworks.shared import (
    MAG7_SYMBOLS,
    OHLCV_COLUMNS,
    combine_equity_curves,
    equal_notional_capital,
    load_price_frame,
)
from quant_orchestrator.strategy import summarize_equity

## Setup

The train window is 2020-2025 and the forward window starts in 2026. The same global provider/symbol date intersection is used before splitting so provider differences are not caused by mismatched calendars. SMA values are computed by Quant Warehouse on the full available history up to each period end, then filtered to the period being tested.

In [2]:
SYMBOLS = MAG7_SYMBOLS
PROVIDERS = ("yfinance", "fmp")
FORWARD_FRAMEWORKS = ("zipline", "nautilus")
BASELINE_FORWARD_FRAMEWORK = FORWARD_FRAMEWORKS[0]
OPTIMIZATION_FRAMEWORK = "backtesting.py"
CAPITAL_BASE = 100_000.0
TRAIN_START = "2020-01-01"
TRAIN_END = "2025-12-31"
TEST_START = "2026-01-01"
TEST_END = None
PRICE_BASIS = "warehouse_splits_and_dividends_adjusted_ohlc"
DATE_ALIGNMENT = "global_provider_symbol_intersection"
MIN_TRAIN_RETURN = 1.0
MIN_TRAIN_SHARPE = 0.5
MIN_TRAIN_TRADES = 20
TOP_PARAMS_PER_PROVIDER = 3
MVO_RANDOM_PORTFOLIOS = 20_000
MAX_STRATEGY_WEIGHT = 0.35
MIN_ACTIVE_WEIGHT = 0.01
RISK_AVERSION = 8.0

WINDOW_SPEC = WindowSpec(
    mode="fixed",
    train_start=TRAIN_START,
    train_end=TRAIN_END,
    test_start=TEST_START,
    test_end=TEST_END,
)
WINDOW = build_walk_forward_windows(WINDOW_SPEC)[0]

CONFIG = pd.DataFrame(
    [
        {
            "symbols": ", ".join(SYMBOLS),
            "providers": ", ".join(PROVIDERS),
            "optimizer": OPTIMIZATION_FRAMEWORK,
            "forward_frameworks": ", ".join(FORWARD_FRAMEWORKS),
            "baseline_forward_framework": BASELINE_FORWARD_FRAMEWORK,
            "train_start": str(WINDOW.train_start.date()),
            "train_end": str(WINDOW.train_end.date()),
            "test_start": str(WINDOW.test_start.date()),
            "test_end": "latest common date",
            "capital_base": CAPITAL_BASE,
            "fast_grid": ", ".join(map(str, FAST_WINDOWS)),
            "slow_grid": ", ".join(map(str, SLOW_WINDOWS)),
            "price_basis": PRICE_BASIS,
            "date_alignment": DATE_ALIGNMENT,
            "candidate_filter": f"return >= {MIN_TRAIN_RETURN}, sharpe >= {MIN_TRAIN_SHARPE}, trades >= {MIN_TRAIN_TRADES}",
            "top_params_per_provider": TOP_PARAMS_PER_PROVIDER,
            "max_strategy_weight": MAX_STRATEGY_WEIGHT,
        }
    ]
)
display(CONFIG.T.rename(columns={0: "value"}))

pipeline_context = PipelineContext(metadata={"notebook": "sample_strategy_validation"})
_ = pipeline_context.set("config", CONFIG)

,value
symbols,"AAPL, AMZN, GOOGL, META, MSFT, NVDA, TSLA"
providers,"yfinance, fmp"
optimizer,backtesting.py
forward_frameworks,"zipline, nautilus"
baseline_forward_framework,zipline
train_start,2020-01-01
train_end,2025-12-31
test_start,2026-01-01
test_end,latest common date
capital_base,100000.0


In [3]:
def _normalize_index(frame: pd.DataFrame) -> pd.DataFrame:
    normalized = frame.rename(columns=str.lower).copy()
    normalized.index = pd.DatetimeIndex(normalized.index)
    if normalized.index.tz is not None:
        normalized.index = normalized.index.tz_convert(None)
    return normalized.sort_index()


def load_all_provider_prices() -> dict[str, dict[str, pd.DataFrame]]:
    raw: dict[str, dict[str, pd.DataFrame]] = {}
    for provider in PROVIDERS:
        raw[provider] = {}
        for symbol in SYMBOLS:
            frame = load_price_frame(symbol, provider=provider, start=TRAIN_START, end=TEST_END)
            raw[provider][symbol] = _normalize_index(frame).loc[:, list(OHLCV_COLUMNS)].copy()
    return raw


def align_provider_symbol_frames(raw: dict[str, dict[str, pd.DataFrame]]) -> tuple[dict[str, dict[str, pd.DataFrame]], pd.DataFrame]:
    common_index: pd.DatetimeIndex | None = None
    for provider in PROVIDERS:
        for symbol in SYMBOLS:
            index = pd.DatetimeIndex(raw[provider][symbol].index).normalize()
            common_index = index if common_index is None else common_index.intersection(index)
    if common_index is None or common_index.empty:
        raise ValueError("No common provider/symbol calendar found")
    common_index = pd.DatetimeIndex(common_index).sort_values()

    aligned: dict[str, dict[str, pd.DataFrame]] = {}
    audit_rows = []
    for provider in PROVIDERS:
        aligned[provider] = {}
        for symbol in SYMBOLS:
            original = raw[provider][symbol]
            frame = original.copy()
            frame.index = pd.DatetimeIndex(frame.index).normalize()
            frame = frame.loc[frame.index.isin(common_index)].copy()
            aligned[provider][symbol] = frame
            audit_rows.append(
                {
                    "provider": provider,
                    "symbol": symbol,
                    "raw_start": original.index.min().date(),
                    "raw_end": original.index.max().date(),
                    "raw_rows": len(original),
                    "aligned_start": frame.index.min().date(),
                    "aligned_end": frame.index.max().date(),
                    "aligned_rows": len(frame),
                    "dropped_rows": len(original) - len(frame),
                }
            )
    return aligned, pd.DataFrame(audit_rows)


def build_period_signal_frame(
    prices: pd.DataFrame,
    *,
    fast_window: int,
    slow_window: int,
    start: pd.Timestamp,
    end: pd.Timestamp,
) -> pd.DataFrame:
    feature_source = prices.loc[:end].copy()
    signal_frame = build_sma_frame(feature_source, fast_window=fast_window, slow_window=slow_window)
    period = signal_frame.loc[start:end].copy()
    if period.empty:
        raise ValueError(f"No signal rows for {start.date()} to {end.date()}")
    period["signal"] = period["signal"].fillna(0).astype(int)
    return period


def performance_score(summary: dict[str, float | str]) -> float:
    return float(summary["total_return"]) + float(summary["max_drawdown"])


def summarize_portfolio_run(
    *,
    framework: str,
    symbol: str,
    equity: pd.Series,
    elapsed_seconds: float,
    bars: int,
    trades: int,
) -> pd.DataFrame:
    return build_common_summary(
        framework=framework,
        symbol=symbol,
        equity=equity,
        elapsed_seconds=elapsed_seconds,
        bars=bars,
        trades=trades,
    )

In [4]:
def run_nautilus_signal_isolated(
    frame: pd.DataFrame,
    *,
    symbol: str,
    capital_base: float,
) -> FrameworkRun:
    payload = {
        "frame_json": frame.to_json(orient="split", date_format="iso"),
        "symbol": symbol,
        "capital_base": capital_base,
    }
    code = r"""
from __future__ import annotations

import json
import sys
from io import StringIO
from pathlib import Path

import pandas as pd

repo_root = Path.cwd().resolve()
if not (repo_root / "quant_orchestrator").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from quant_orchestrator.backtests.research import run_nautilus

payload = json.loads(sys.stdin.read())
frame = pd.read_json(StringIO(payload["frame_json"]), orient="split")
frame.index = pd.DatetimeIndex(pd.to_datetime(frame.index))
run = run_nautilus(frame, symbol=payload["symbol"], capital_base=float(payload["capital_base"]))
row = run.summary.iloc[0].to_dict()
clean = {}
for key, value in row.items():
    if hasattr(value, "item"):
        try:
            value = value.item()
        except Exception:
            pass
    if hasattr(value, "isoformat"):
        try:
            value = value.isoformat()
        except Exception:
            pass
    clean[key] = value
print(json.dumps(clean, default=str))
print("__EQUITY__")
print(run.equity.to_json(date_format="iso"))
"""
    completed = subprocess.run(
        [sys.executable, "-c", code],
        check=True,
        capture_output=True,
        text=True,
        cwd=Path.cwd(),
        input=json.dumps(payload),
    )
    stdout = completed.stdout.strip().splitlines()
    if "__EQUITY__" not in stdout:
        raise RuntimeError(
            "Nautilus isolated run did not emit an equity payload:\n"
            + completed.stdout
            + "\n"
            + completed.stderr
        )
    marker = stdout.index("__EQUITY__")
    row = json.loads(stdout[marker - 1])
    equity = pd.read_json(StringIO(stdout[marker + 1]), typ="series")
    equity.index = pd.DatetimeIndex(pd.to_datetime(equity.index))
    return FrameworkRun(raw_result=pd.DataFrame([row]), summary=pd.DataFrame([row]), equity=equity.rename("portfolio_value"))


def run_symbol_with_framework(
    framework: str,
    frame: pd.DataFrame,
    *,
    symbol: str,
    capital_base: float,
) -> FrameworkRun:
    if framework == "zipline":
        return run_zipline(frame, symbol=symbol, capital_base=capital_base)
    if framework == "nautilus":
        return run_nautilus_signal_isolated(frame, symbol=symbol, capital_base=capital_base)
    raise ValueError(f"Unsupported forward framework: {framework}")

In [5]:
def annualized_sharpe(equity: pd.Series) -> float:
    returns = equity.pct_change().dropna()
    if returns.empty or float(returns.std()) == 0.0:
        return 0.0
    return float((returns.mean() / returns.std()) * np.sqrt(252))


def run_backtesting_py_portfolio_equity(
    provider_prices: dict[str, pd.DataFrame],
    *,
    fast_window: int,
    slow_window: int,
    start: pd.Timestamp,
    end: pd.Timestamp,
) -> pd.Series:
    capital_per_symbol = equal_notional_capital(CAPITAL_BASE, len(SYMBOLS))
    sleeves = []
    for symbol in SYMBOLS:
        signal_frame = build_period_signal_frame(
            provider_prices[symbol],
            fast_window=fast_window,
            slow_window=slow_window,
            start=start,
            end=end,
        )
        run = run_backtesting_py(signal_frame, symbol=symbol, capital_base=capital_per_symbol)
        sleeves.append(run.equity.rename(symbol))
    return combine_equity_curves(sleeves).rename("portfolio_value")


def optimize_provider_with_backtesting_py(
    provider: str,
    provider_prices: dict[str, pd.DataFrame],
) -> tuple[pd.DataFrame, pd.Series, dict[tuple[int, int], pd.Series]]:
    rows = []
    train_equities: dict[tuple[int, int], pd.Series] = {}
    capital_per_symbol = equal_notional_capital(CAPITAL_BASE, len(SYMBOLS))
    optimizer_started = perf_counter()
    train_start = WINDOW.train_start
    train_end = WINDOW.train_end

    for fast_window in FAST_WINDOWS:
        for slow_window in SLOW_WINDOWS:
            if fast_window >= slow_window:
                continue
            combo_started = perf_counter()
            sleeves = []
            trades = 0
            symbol_bars = 0
            for symbol in SYMBOLS:
                signal_frame = build_period_signal_frame(
                    provider_prices[symbol],
                    fast_window=int(fast_window),
                    slow_window=int(slow_window),
                    start=train_start,
                    end=train_end,
                )
                run = run_backtesting_py(signal_frame, symbol=symbol, capital_base=capital_per_symbol)
                sleeves.append(run.equity.rename(symbol))
                trades += int(run.summary["trades"].iloc[0])
                symbol_bars += len(signal_frame)

            portfolio_equity = combine_equity_curves(sleeves).rename("portfolio_value")
            train_equities[(int(fast_window), int(slow_window))] = portfolio_equity
            summary = summarize_equity(portfolio_equity)
            rows.append(
                {
                    "provider": provider,
                    "optimization_framework": OPTIMIZATION_FRAMEWORK,
                    "fast_window": int(fast_window),
                    "slow_window": int(slow_window),
                    "train_start": str(train_start.date()),
                    "train_end": str(train_end.date()),
                    "bars": int(symbol_bars),
                    "trades": trades,
                    "final_equity": summary["final_equity"],
                    "total_return": summary["total_return"],
                    "max_drawdown": summary["max_drawdown"],
                    "daily_vol": summary["daily_vol"],
                    "train_sharpe": annualized_sharpe(portfolio_equity),
                    "performance_score": performance_score(summary),
                    "combo_seconds": round(perf_counter() - combo_started, 4),
                }
            )

    grid = pd.DataFrame(rows).sort_values(
        by=["performance_score", "total_return", "train_sharpe", "max_drawdown"],
        ascending=[False, False, False, False],
    ).reset_index(drop=True)
    grid["train_rank"] = grid.index + 1
    grid["optimizer_wall_clock_seconds"] = round(perf_counter() - optimizer_started, 4)
    best = grid.iloc[0]
    best_equity = train_equities[(int(best["fast_window"]), int(best["slow_window"]))]
    return grid, best_equity, train_equities


def forward_test_provider_framework(
    *,
    provider: str,
    framework: str,
    provider_prices: dict[str, pd.DataFrame],
    fast_window: int,
    slow_window: int,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series]:
    capital_per_symbol = equal_notional_capital(CAPITAL_BASE, len(SYMBOLS))
    test_end = max(frame.index.max() for frame in provider_prices.values())
    started = perf_counter()
    sleeves = []
    symbol_rows = []
    trades = 0
    symbol_bars = 0

    for symbol in SYMBOLS:
        signal_frame = build_period_signal_frame(
            provider_prices[symbol],
            fast_window=fast_window,
            slow_window=slow_window,
            start=WINDOW.test_start,
            end=test_end,
        )
        run = run_symbol_with_framework(
            framework,
            signal_frame,
            symbol=symbol,
            capital_base=capital_per_symbol,
        )
        symbol_summary = run.summary.copy()
        symbol_summary["provider"] = provider
        symbol_summary["forward_framework"] = framework
        symbol_summary["fast_window"] = fast_window
        symbol_summary["slow_window"] = slow_window
        symbol_rows.append(symbol_summary)
        sleeves.append(run.equity.rename(symbol))
        trades += int(run.summary["trades"].iloc[0])
        symbol_bars += len(signal_frame)

    portfolio_equity = combine_equity_curves(sleeves)
    elapsed = perf_counter() - started
    portfolio_summary = summarize_portfolio_run(
        framework=framework,
        symbol="MAG7",
        equity=portfolio_equity,
        elapsed_seconds=elapsed,
        bars=len(portfolio_equity),
        trades=trades,
    )
    portfolio_summary["provider"] = provider
    portfolio_summary["forward_framework"] = framework
    portfolio_summary["optimization_framework"] = OPTIMIZATION_FRAMEWORK
    portfolio_summary["fast_window"] = fast_window
    portfolio_summary["slow_window"] = slow_window
    portfolio_summary["symbol_bars"] = symbol_bars
    return portfolio_summary, pd.concat(symbol_rows, ignore_index=True), portfolio_equity.rename("portfolio_value")

## Data Jobs

The first job loads both providers from Quant Warehouse and aligns them to a common provider/symbol calendar. This keeps the WFO comparison from being polluted by different start/end dates or missing sessions.

In [6]:
raw_price_frames = load_all_provider_prices()
price_frames_by_provider, date_alignment_audit = align_provider_symbol_frames(raw_price_frames)

display(date_alignment_audit)
_ = pipeline_context.update({"raw_price_frames": raw_price_frames, "price_frames_by_provider": price_frames_by_provider, "date_alignment_audit": date_alignment_audit})

period_audit_rows = []
for provider in PROVIDERS:
    for symbol in SYMBOLS:
        frame = price_frames_by_provider[provider][symbol]
        period_audit_rows.append(
            {
                "provider": provider,
                "symbol": symbol,
                "train_rows": len(frame.loc[WINDOW.train_start:WINDOW.train_end]),
                "test_rows": len(frame.loc[WINDOW.test_start:]),
                "first_test_date": frame.loc[WINDOW.test_start:].index.min().date(),
                "last_test_date": frame.loc[WINDOW.test_start:].index.max().date(),
            }
        )
period_audit = pd.DataFrame(period_audit_rows)
display(period_audit)

,provider,symbol,raw_start,raw_end,raw_rows,aligned_start,aligned_end,aligned_rows,dropped_rows
0,yfinance,AAPL,2020-01-02,2026-06-26,1629,2020-01-02,2026-06-24,1627,2
1,yfinance,AMZN,2020-01-02,2026-06-26,1629,2020-01-02,2026-06-24,1627,2
2,yfinance,GOOGL,2020-01-02,2026-06-26,1629,2020-01-02,2026-06-24,1627,2
3,yfinance,META,2020-01-02,2026-06-26,1629,2020-01-02,2026-06-24,1627,2
4,yfinance,MSFT,2020-01-02,2026-06-26,1629,2020-01-02,2026-06-24,1627,2
5,yfinance,NVDA,2020-01-02,2026-06-26,1629,2020-01-02,2026-06-24,1627,2
6,yfinance,TSLA,2020-01-02,2026-06-26,1629,2020-01-02,2026-06-24,1627,2
7,fmp,AAPL,2020-01-02,2026-06-24,1627,2020-01-02,2026-06-24,1627,0
8,fmp,AMZN,2020-01-02,2026-06-24,1627,2020-01-02,2026-06-24,1627,0
9,fmp,GOOGL,2020-01-02,2026-06-24,1627,2020-01-02,2026-06-24,1627,0


,provider,symbol,train_rows,test_rows,first_test_date,last_test_date
0,yfinance,AAPL,1508,119,2026-01-02,2026-06-24
1,yfinance,AMZN,1508,119,2026-01-02,2026-06-24
2,yfinance,GOOGL,1508,119,2026-01-02,2026-06-24
3,yfinance,META,1508,119,2026-01-02,2026-06-24
4,yfinance,MSFT,1508,119,2026-01-02,2026-06-24
5,yfinance,NVDA,1508,119,2026-01-02,2026-06-24
6,yfinance,TSLA,1508,119,2026-01-02,2026-06-24
7,fmp,AAPL,1508,119,2026-01-02,2026-06-24
8,fmp,AMZN,1508,119,2026-01-02,2026-06-24
9,fmp,GOOGL,1508,119,2026-01-02,2026-06-24


## Optimization Jobs

For each data provider, `backtesting.py` searches the same SMA grid on the MAG7 portfolio in-sample. The selected parameter pair is provider-specific and is not shared across vendors.

In [7]:
optimization_grids = {}
train_equity_by_provider = {}
train_combo_equity_by_provider = {}
optimization_top_tables = []

for provider in PROVIDERS:
    grid, train_equity, train_combo_equity = optimize_provider_with_backtesting_py(provider, price_frames_by_provider[provider])
    optimization_grids[provider] = grid
    train_equity_by_provider[provider] = train_equity
    train_combo_equity_by_provider[provider] = train_combo_equity
    optimization_top_tables.append(grid.head(10))

optimization_top10 = pd.concat(optimization_top_tables, ignore_index=True)
selected_parameters = pd.concat([grid.head(1) for grid in optimization_grids.values()], ignore_index=True)

display(optimization_top10)
display(selected_parameters)

Loading BokehJS ...

,provider,optimization_framework,fast_window,slow_window,train_start,train_end,bars,trades,final_equity,total_return,max_drawdown,daily_vol,train_sharpe,performance_score,combo_seconds,train_rank,optimizer_wall_clock_seconds
0,yfinance,backtesting.py,35,60,2020-01-01,2025-12-31,10556,89,280227.82,1.8023,-0.2882,0.0089,1.288688,1.5141,0.4108,1,32.1673
1,yfinance,backtesting.py,40,60,2020-01-01,2025-12-31,10556,97,273981.22,1.7398,-0.3020,0.0091,1.235225,1.4378,0.2855,2,32.1673
2,yfinance,backtesting.py,30,60,2020-01-01,2025-12-31,10556,94,271950.97,1.7195,-0.3005,0.0091,1.233295,1.4190,0.2862,3,32.1673
3,yfinance,backtesting.py,15,120,2020-01-01,2025-12-31,10556,60,261171.18,1.6117,-0.2138,0.0106,1.034229,1.3979,0.4087,4,32.1673
4,yfinance,backtesting.py,20,140,2020-01-01,2025-12-31,10556,37,248400.07,1.4840,-0.1330,0.0087,1.170127,1.3510,0.4138,5,32.1673
5,yfinance,backtesting.py,25,120,2020-01-01,2025-12-31,10556,49,252911.89,1.5291,-0.2098,0.0100,1.052792,1.3193,0.2907,6,32.1673
6,yfinance,backtesting.py,35,70,2020-01-01,2025-12-31,10556,66,259663.85,1.5966,-0.2904,0.0095,1.135527,1.3062,0.2896,7,32.1673
7,yfinance,backtesting.py,50,60,2020-01-01,2025-12-31,10556,116,257090.83,1.5709,-0.2693,0.0092,1.159596,1.3016,0.4094,8,32.1673
8,yfinance,backtesting.py,45,60,2020-01-01,2025-12-31,10556,106,256653.93,1.5665,-0.2720,0.0094,1.134211,1.2945,0.2886,9,32.1673
9,yfinance,backtesting.py,45,100,2020-01-01,2025-12-31,10556,49,248101.03,1.4810,-0.2132,0.0095,1.084442,1.2678,0.2852,10,32.1673


,provider,optimization_framework,fast_window,slow_window,train_start,train_end,bars,trades,final_equity,total_return,max_drawdown,daily_vol,train_sharpe,performance_score,combo_seconds,train_rank,optimizer_wall_clock_seconds
0,yfinance,backtesting.py,35,60,2020-01-01,2025-12-31,10556,89,280227.82,1.8023,-0.2882,0.0089,1.288688,1.5141,0.4108,1,32.1673
1,fmp,backtesting.py,35,60,2020-01-01,2025-12-31,10556,89,278559.77,1.7856,-0.2885,0.0089,1.280824,1.4971,0.4117,1,32.0458


## Forward-Test Jobs

The forward jobs use the selected provider-specific parameters, then run the 2026 out-of-sample MAG7 portfolio separately through Zipline Reloaded and NautilusTrader. The frameworks consume the same in-memory signal frames for a given provider.

In [8]:
forward_rows = []
forward_symbol_rows = []
forward_equity = {}
forward_summary_by_strategy = {}

for provider in PROVIDERS:
    best = selected_parameters[selected_parameters["provider"] == provider].iloc[0]
    fast_window = int(best["fast_window"])
    slow_window = int(best["slow_window"])
    for framework in FORWARD_FRAMEWORKS:
        portfolio_summary, symbol_summary, equity = forward_test_provider_framework(
            provider=provider,
            framework=framework,
            provider_prices=price_frames_by_provider[provider],
            fast_window=fast_window,
            slow_window=slow_window,
        )
        portfolio_summary["train_total_return"] = float(best["total_return"])
        portfolio_summary["train_max_drawdown"] = float(best["max_drawdown"])
        portfolio_summary["train_sharpe"] = float(best["train_sharpe"])
        portfolio_summary["train_performance_score"] = float(best["performance_score"])
        strategy_key = (provider, framework, fast_window, slow_window)
        forward_rows.append(portfolio_summary)
        forward_symbol_rows.append(symbol_summary)
        forward_equity[strategy_key] = equity
        forward_summary_by_strategy[strategy_key] = portfolio_summary.iloc[0].to_dict()

forward_report = pd.concat(forward_rows, ignore_index=True)
forward_symbol_report = pd.concat(forward_symbol_rows, ignore_index=True)
slowest_wall_clock = float(forward_report["elapsed_seconds"].max())
forward_report["speedup_vs_slowest"] = slowest_wall_clock / forward_report["elapsed_seconds"]
forward_report["performance_score"] = forward_report["total_return"] + forward_report["max_drawdown"]
forward_report = forward_report.sort_values(
    by=["provider", "forward_framework"],
).reset_index(drop=True)

display(forward_report)
display(forward_symbol_report.sort_values(["provider", "forward_framework", "symbol"]).reset_index(drop=True))

,framework,symbol,start,end,bars,trades,initial_equity,final_equity,total_return,annualized_return,...,optimization_framework,fast_window,slow_window,symbol_bars,train_total_return,train_max_drawdown,train_sharpe,train_performance_score,speedup_vs_slowest,performance_score
0,nautilus,MAG7,2026-01-02,2026-06-24,119,24,100000.00,97362.41,-0.0264,-0.0550,...,backtesting.py,35,60,833,1.7856,-0.2885,1.280824,1.4971,1.016838,-0.0564
1,zipline,MAG7,2026-01-02,2026-06-24,119,30,100000.00,97151.20,-0.0285,-0.0594,...,backtesting.py,35,60,833,1.7856,-0.2885,1.280824,1.4971,37.573879,-0.0606
2,nautilus,MAG7,2026-01-02,2026-06-24,119,24,99999.97,97382.77,-0.0262,-0.0546,...,backtesting.py,35,60,833,1.8023,-0.2882,1.288688,1.5141,1.000000,-0.0559
3,zipline,MAG7,2026-01-02,2026-06-24,119,30,100000.00,97555.56,-0.0244,-0.0511,...,backtesting.py,35,60,833,1.8023,-0.2882,1.288688,1.5141,23.726034,-0.0525


,framework,symbol,start,end,bars,trades,initial_equity,final_equity,total_return,annualized_return,...,sharpe,calmar,elapsed_seconds,bars_per_second,native_last_value,provider,forward_framework,fast_window,slow_window,native_fills
0,nautilus,AAPL,2026-01-02,2026-06-24,119,5,14285.71,13978.91,-0.0215,-0.0449,...,-1.3074,-1.6738,0.0126,9445.43,13978.914286,fmp,nautilus,35,60,5.0
1,nautilus,AMZN,2026-01-02,2026-06-24,119,3,14285.71,13613.26,-0.0471,-0.0971,...,-1.7358,-1.8571,0.0108,11023.52,13613.264286,fmp,nautilus,35,60,3.0
2,nautilus,GOOGL,2026-01-02,2026-06-24,119,3,14285.71,14106.96,-0.0125,-0.0263,...,-0.3421,-0.6149,0.0117,10201.07,14106.964286,fmp,nautilus,35,60,3.0
3,nautilus,META,2026-01-02,2026-06-24,119,6,14285.71,13897.21,-0.0272,-0.0567,...,-0.9137,-0.8620,0.0125,9535.62,13897.214286,fmp,nautilus,35,60,6.0
4,nautilus,MSFT,2026-01-02,2026-06-24,119,1,14285.71,13989.54,-0.0207,-0.0434,...,-1.2174,-0.9557,0.0104,11417.45,13989.544286,fmp,nautilus,35,60,1.0
5,nautilus,NVDA,2026-01-02,2026-06-24,119,3,14285.71,13997.35,-0.0202,-0.0423,...,-0.4926,-0.9368,0.0103,11531.38,13997.354286,fmp,nautilus,35,60,3.0
6,nautilus,TSLA,2026-01-02,2026-06-24,119,3,14285.71,13779.15,-0.0355,-0.0736,...,-1.1433,-1.7218,0.0110,10771.81,13779.154286,fmp,nautilus,35,60,3.0
7,zipline,AAPL,2026-01-02,2026-06-24,119,6,14285.71,14026.46,-0.0181,-0.0380,...,-1.1026,-1.5565,0.0266,4477.05,14026.460096,fmp,zipline,35,60,NaN
8,zipline,AMZN,2026-01-02,2026-06-24,119,4,14285.71,13398.69,-0.0621,-0.1269,...,-2.3150,-1.9944,0.1862,638.99,13398.692361,fmp,zipline,35,60,NaN
9,zipline,GOOGL,2026-01-02,2026-06-24,119,4,14285.71,13688.69,-0.0418,-0.0864,...,-1.4864,-1.4100,0.0263,4517.86,13688.691376,fmp,zipline,35,60,NaN


## Multi-Strategy Portfolio Optimization

This job keeps the single-best forward test as the benchmark, then asks whether a filtered portfolio of several parameter/vendor/framework strategies is better out of sample. Candidate strategies must pass common in-sample filters before mean-variance optimization. The final active portfolio must include more than one data vendor and more than one backtesting framework.

In [9]:
def build_candidate_universe() -> pd.DataFrame:
    rows = []
    for provider, grid in optimization_grids.items():
        eligible = filter_results(
            grid,
            [
                MetricFilter("total_return", ">=", MIN_TRAIN_RETURN),
                MetricFilter("train_sharpe", ">=", MIN_TRAIN_SHARPE),
                MetricFilter("trades", ">=", MIN_TRAIN_TRADES),
            ],
        ).head(TOP_PARAMS_PER_PROVIDER)
        if eligible.empty:
            eligible = grid.head(TOP_PARAMS_PER_PROVIDER)
        for row in eligible.itertuples(index=False):
            for framework in FORWARD_FRAMEWORKS:
                rows.append(
                    {
                        "strategy_id": f"{provider}|{framework}|{int(row.fast_window)}|{int(row.slow_window)}",
                        "provider": provider,
                        "forward_framework": framework,
                        "fast_window": int(row.fast_window),
                        "slow_window": int(row.slow_window),
                        "train_rank": int(row.train_rank),
                        "train_total_return": float(row.total_return),
                        "train_max_drawdown": float(row.max_drawdown),
                        "train_sharpe": float(row.train_sharpe),
                        "train_trades": int(row.trades),
                        "train_performance_score": float(row.performance_score),
                    }
                )
    candidates = pd.DataFrame(rows)
    if candidates["provider"].nunique() < 2 or candidates["forward_framework"].nunique() < 2:
        raise ValueError("Candidate universe must include at least two providers and two frameworks")
    return candidates


def build_candidate_train_returns(candidates: pd.DataFrame) -> pd.DataFrame:
    curves = {
        row.strategy_id: train_combo_equity_by_provider[row.provider][(int(row.fast_window), int(row.slow_window))]
        for row in candidates.itertuples(index=False)
    }
    return returns_matrix_from_equity_curves(curves).dropna()


def optimize_strategy_weights(candidates: pd.DataFrame, train_returns: pd.DataFrame) -> pd.DataFrame:
    providers = candidates.set_index("strategy_id").loc[train_returns.columns, "provider"]
    frameworks = candidates.set_index("strategy_id").loc[train_returns.columns, "forward_framework"]
    result = optimize_random_mean_variance_weights(
        train_returns,
        iterations=MVO_RANDOM_PORTFOLIOS,
        max_weight=MAX_STRATEGY_WEIGHT,
        min_active_weight=MIN_ACTIVE_WEIGHT,
        risk_aversion=RISK_AVERSION,
        seed=20260629,
        group_constraints={"provider": providers, "framework": frameworks},
    )
    weights = candidates.copy()
    weights = weights.merge(result.weights.rename("weight"), left_on="strategy_id", right_index=True, how="left")
    weights["weight"] = weights["weight"].fillna(0.0)
    weights["active"] = weights["weight"] >= MIN_ACTIVE_WEIGHT
    weights["mvo_score"] = result.score
    weights["accepted_random_portfolios"] = result.accepted_portfolios
    return weights.sort_values("weight", ascending=False).reset_index(drop=True)


def run_strategy_forward_once(row) -> tuple[pd.DataFrame, pd.Series]:
    key = (row.provider, row.forward_framework, int(row.fast_window), int(row.slow_window))
    if key not in forward_equity:
        portfolio_summary, _, equity = forward_test_provider_framework(
            provider=row.provider,
            framework=row.forward_framework,
            provider_prices=price_frames_by_provider[row.provider],
            fast_window=int(row.fast_window),
            slow_window=int(row.slow_window),
        )
        portfolio_summary["train_total_return"] = float(row.train_total_return)
        portfolio_summary["train_max_drawdown"] = float(row.train_max_drawdown)
        portfolio_summary["train_sharpe"] = float(row.train_sharpe)
        portfolio_summary["train_performance_score"] = float(row.train_performance_score)
        forward_equity[key] = equity
        forward_summary_by_strategy[key] = portfolio_summary.iloc[0].to_dict()
    return pd.DataFrame([forward_summary_by_strategy[key]]), forward_equity[key]


def run_weighted_forward_portfolio(weights: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series]:
    active_weights = weights[weights["active"]].copy()
    active_weights["weight"] = active_weights["weight"] / active_weights["weight"].sum()
    started = perf_counter()
    weighted_curves = []
    strategy_rows = []
    total_trades = 0

    for row in active_weights.itertuples(index=False):
        summary, equity = run_strategy_forward_once(row)
        equity = equity.copy()
        equity.index = pd.DatetimeIndex(pd.to_datetime(equity.index))
        if equity.index.tz is not None:
            equity.index = equity.index.tz_convert(None)
        equity = equity.sort_index()
        summary = summary.copy()
        summary["strategy_id"] = row.strategy_id
        summary["weight"] = float(row.weight)
        strategy_rows.append(summary)
        total_trades += int(summary["trades"].iloc[0])
        weighted_curves.append((equity / float(equity.iloc[0]) * (CAPITAL_BASE * float(row.weight))).rename(row.strategy_id))

    combined_index = pd.Index([])
    for curve in weighted_curves:
        combined_index = combined_index.union(curve.index)
    portfolio_equity = pd.Series(0.0, index=combined_index, name="portfolio_value")
    for curve in weighted_curves:
        portfolio_equity = portfolio_equity.add(curve.reindex(combined_index).ffill().fillna(curve.iloc[0]), fill_value=0.0)
    portfolio_equity = portfolio_equity.sort_index().rename("portfolio_value")

    portfolio_equity = normalize_equity_curve(portfolio_equity)
    portfolio_summary = build_common_summary(
        framework="mean_variance_strategy_portfolio",
        symbol="MAG7",
        equity=portfolio_equity,
        elapsed_seconds=perf_counter() - started,
        bars=len(portfolio_equity),
        trades=total_trades,
    )
    portfolio_summary["provider"] = "multi_vendor"
    portfolio_summary["forward_framework"] = "multi_framework"
    portfolio_summary["optimization_framework"] = OPTIMIZATION_FRAMEWORK
    portfolio_summary["active_strategies"] = len(active_weights)
    portfolio_summary["active_providers"] = active_weights["provider"].nunique()
    portfolio_summary["active_frameworks"] = active_weights["forward_framework"].nunique()
    portfolio_summary["performance_score"] = portfolio_summary["total_return"] + portfolio_summary["max_drawdown"]
    return portfolio_summary, pd.concat(strategy_rows, ignore_index=True), portfolio_equity


candidate_universe = build_candidate_universe()
candidate_train_returns = build_candidate_train_returns(candidate_universe)
strategy_weights = optimize_strategy_weights(candidate_universe, candidate_train_returns)
weighted_forward_report, weighted_strategy_forward_report, weighted_forward_equity = run_weighted_forward_portfolio(strategy_weights)

baseline_provider = selected_parameters.sort_values(
    by=["performance_score", "total_return", "train_sharpe"],
    ascending=[False, False, False],
).iloc[0]["provider"]
baseline_forward_report = forward_report[
    (forward_report["provider"] == baseline_provider)
    & (forward_report["forward_framework"] == BASELINE_FORWARD_FRAMEWORK)
].copy()
if baseline_forward_report.empty:
    raise ValueError("No predeclared single-strategy benchmark row found")
baseline_forward_report["approach"] = "in_sample_selected_single_strategy"
weighted_comparison_row = weighted_forward_report.copy()
weighted_comparison_row["approach"] = "filtered_mean_variance_portfolio"
strategy_portfolio_comparison = pd.concat([baseline_forward_report, weighted_comparison_row], ignore_index=True, sort=False)
strategy_portfolio_comparison = strategy_portfolio_comparison.reset_index(drop=True)

display(candidate_universe)
display(strategy_weights)
display(weighted_strategy_forward_report.sort_values("weight", ascending=False).reset_index(drop=True))
display(strategy_portfolio_comparison)

,strategy_id,provider,forward_framework,fast_window,slow_window,train_rank,train_total_return,train_max_drawdown,train_sharpe,train_trades,train_performance_score
0,yfinance|zipline|35|60,yfinance,zipline,35,60,1,1.8023,-0.2882,1.288688,89,1.5141
1,yfinance|nautilus|35|60,yfinance,nautilus,35,60,1,1.8023,-0.2882,1.288688,89,1.5141
2,yfinance|zipline|40|60,yfinance,zipline,40,60,2,1.7398,-0.3020,1.235225,97,1.4378
3,yfinance|nautilus|40|60,yfinance,nautilus,40,60,2,1.7398,-0.3020,1.235225,97,1.4378
4,yfinance|zipline|30|60,yfinance,zipline,30,60,3,1.7195,-0.3005,1.233295,94,1.4190
5,yfinance|nautilus|30|60,yfinance,nautilus,30,60,3,1.7195,-0.3005,1.233295,94,1.4190
6,fmp|zipline|35|60,fmp,zipline,35,60,1,1.7856,-0.2885,1.280824,89,1.4971
7,fmp|nautilus|35|60,fmp,nautilus,35,60,1,1.7856,-0.2885,1.280824,89,1.4971
8,fmp|zipline|40|60,fmp,zipline,40,60,2,1.7360,-0.3026,1.233960,98,1.4334
9,fmp|nautilus|40|60,fmp,nautilus,40,60,2,1.7360,-0.3026,1.233960,98,1.4334


,strategy_id,provider,forward_framework,fast_window,slow_window,train_rank,train_total_return,train_max_drawdown,train_sharpe,train_trades,train_performance_score,weight,active,mvo_score,accepted_random_portfolios
0,yfinance|nautilus|35|60,yfinance,nautilus,35,60,1,1.8023,-0.2882,1.288688,89,1.5141,0.346245,True,0.021438,17961
1,yfinance|zipline|35|60,yfinance,zipline,35,60,1,1.8023,-0.2882,1.288688,89,1.5141,0.340046,True,0.021438,17961
2,fmp|zipline|40|60,fmp,zipline,40,60,2,1.7360,-0.3026,1.233960,98,1.4334,0.092870,True,0.021438,17961
3,yfinance|nautilus|30|60,yfinance,nautilus,30,60,3,1.7195,-0.3005,1.233295,94,1.4190,0.079128,True,0.021438,17961
4,fmp|zipline|35|60,fmp,zipline,35,60,1,1.7856,-0.2885,1.280824,89,1.4971,0.046300,True,0.021438,17961
5,fmp|nautilus|30|60,fmp,nautilus,30,60,3,1.7181,-0.2985,1.234535,94,1.4196,0.036103,True,0.021438,17961
6,fmp|zipline|30|60,fmp,zipline,30,60,3,1.7181,-0.2985,1.234535,94,1.4196,0.023834,True,0.021438,17961
7,yfinance|zipline|40|60,yfinance,zipline,40,60,2,1.7398,-0.3020,1.235225,97,1.4378,0.023497,True,0.021438,17961
8,yfinance|nautilus|40|60,yfinance,nautilus,40,60,2,1.7398,-0.3020,1.235225,97,1.4378,0.011977,True,0.021438,17961
9,yfinance|zipline|30|60,yfinance,zipline,30,60,3,1.7195,-0.3005,1.233295,94,1.4190,0.000000,False,0.021438,17961


,framework,symbol,start,end,bars,trades,initial_equity,final_equity,total_return,annualized_return,...,optimization_framework,fast_window,slow_window,symbol_bars,train_total_return,train_max_drawdown,train_sharpe,train_performance_score,strategy_id,weight
0,nautilus,MAG7,2026-01-02,2026-06-24,119,24,99999.97,97382.77,-0.0262,-0.0546,...,backtesting.py,35,60,833,1.8023,-0.2882,1.288688,1.5141,yfinance|nautilus|35|60,0.346245
1,zipline,MAG7,2026-01-02,2026-06-24,119,30,100000.00,97555.56,-0.0244,-0.0511,...,backtesting.py,35,60,833,1.8023,-0.2882,1.288688,1.5141,yfinance|zipline|35|60,0.340046
2,zipline,MAG7,2026-01-02,2026-06-24,119,36,100000.00,96803.50,-0.0320,-0.0665,...,backtesting.py,40,60,833,1.7360,-0.3026,1.233960,1.4334,fmp|zipline|40|60,0.092870
3,nautilus,MAG7,2026-01-02,2026-06-24,119,24,99999.97,97286.44,-0.0271,-0.0566,...,backtesting.py,30,60,833,1.7195,-0.3005,1.233295,1.4190,yfinance|nautilus|30|60,0.079128
4,zipline,MAG7,2026-01-02,2026-06-24,119,30,100000.00,97151.20,-0.0285,-0.0594,...,backtesting.py,35,60,833,1.7856,-0.2885,1.280824,1.4971,fmp|zipline|35|60,0.046300
5,nautilus,MAG7,2026-01-02,2026-06-24,119,24,100000.00,97269.98,-0.0273,-0.0569,...,backtesting.py,30,60,833,1.7181,-0.2985,1.234535,1.4196,fmp|nautilus|30|60,0.036103
6,zipline,MAG7,2026-01-02,2026-06-24,119,30,100000.00,97313.77,-0.0269,-0.0560,...,backtesting.py,30,60,833,1.7181,-0.2985,1.234535,1.4196,fmp|zipline|30|60,0.023834
7,zipline,MAG7,2026-01-02,2026-06-24,119,36,100000.00,96825.29,-0.0317,-0.0660,...,backtesting.py,40,60,833,1.7398,-0.3020,1.235225,1.4378,yfinance|zipline|40|60,0.023497
8,nautilus,MAG7,2026-01-02,2026-06-24,119,30,99999.97,96808.81,-0.0319,-0.0664,...,backtesting.py,40,60,833,1.7398,-0.3020,1.235225,1.4378,yfinance|nautilus|40|60,0.011977


,framework,symbol,start,end,bars,trades,initial_equity,final_equity,total_return,annualized_return,...,train_total_return,train_max_drawdown,train_sharpe,train_performance_score,speedup_vs_slowest,performance_score,approach,active_strategies,active_providers,active_frameworks
0,zipline,MAG7,2026-01-02,2026-06-24,119,30,100000.0,97555.56,-0.0244,-0.0511,...,1.8023,-0.2882,1.288688,1.5141,23.726034,-0.0525,in_sample_selected_single_strategy,NaN,NaN,NaN
1,mean_variance_strategy_portfolio,MAG7,2026-01-02,2026-06-24,119,264,100000.0,97343.71,-0.0266,-0.0554,...,NaN,NaN,NaN,NaN,NaN,-0.0567,filtered_mean_variance_portfolio,9.0,2.0,2.0


## Provider vs Framework Differences

This small decomposition compares how much of the variation in the forward-test table is explained by the data vendor versus the backtesting framework. With only two providers and two forward frameworks this is descriptive, not a statistical proof.

In [10]:
def factor_decomposition(table: pd.DataFrame, *, metric: str) -> pd.DataFrame:
    pivot = table.pivot(index="forward_framework", columns="provider", values=metric)
    grand_mean = float(pivot.to_numpy(dtype=float).mean())
    framework_means = pivot.mean(axis=1)
    provider_means = pivot.mean(axis=0)
    framework_ss = float(len(provider_means) * ((framework_means - grand_mean) ** 2).sum())
    provider_ss = float(len(framework_means) * ((provider_means - grand_mean) ** 2).sum())
    total_ss = float(((pivot - grand_mean) ** 2).to_numpy().sum())
    residual_ss = max(0.0, total_ss - framework_ss - provider_ss)
    total = framework_ss + provider_ss + residual_ss
    return pd.DataFrame(
        [
            {
                "metric": metric,
                "framework_ss": framework_ss,
                "provider_ss": provider_ss,
                "residual_ss": residual_ss,
                "framework_share": framework_ss / total if total else 0.0,
                "provider_share": provider_ss / total if total else 0.0,
                "residual_share": residual_ss / total if total else 0.0,
                "dominant_factor": "provider" if provider_ss > framework_ss else "framework",
                "provider_to_framework_ratio": provider_ss / framework_ss if framework_ss else float("inf"),
            }
        ]
    )

factor_report = pd.concat(
    [
        factor_decomposition(forward_report, metric="performance_score"),
        factor_decomposition(forward_report, metric="total_return"),
        factor_decomposition(forward_report, metric="max_drawdown"),
        factor_decomposition(forward_report, metric="elapsed_seconds"),
    ],
    ignore_index=True,
)

display(factor_report)

,metric,framework_ss,provider_ss,residual_ss,framework_share,provider_share,residual_share,dominant_factor,provider_to_framework_ratio
0,performance_score,1.600000e-07,0.000018,0.000014,0.004835,0.558779,4.363856e-01,provider,115.562500
1,total_return,2.250000e-08,0.000005,0.000004,0.002664,0.547203,4.501332e-01,provider,205.444444
2,max_drawdown,6.250000e-08,0.000005,0.000003,0.007709,0.570151,4.221400e-01,provider,73.960000
3,elapsed_seconds,3.802442e+02,0.106831,0.000109,0.999719,0.000281,2.871097e-07,framework,0.000281


## Written Analysis

In [11]:
baseline_forward = baseline_forward_report.iloc[0]
fastest_forward = forward_report.sort_values("elapsed_seconds").iloc[0]
slowest_forward = forward_report.sort_values("elapsed_seconds", ascending=False).iloc[0]
provider_params = selected_parameters.loc[:, ["provider", "fast_window", "slow_window", "total_return", "max_drawdown", "train_sharpe", "performance_score"]]
perf_factor = factor_report[factor_report["metric"] == "performance_score"].iloc[0]
return_factor = factor_report[factor_report["metric"] == "total_return"].iloc[0]
speed_factor = factor_report[factor_report["metric"] == "elapsed_seconds"].iloc[0]
weighted_row = weighted_forward_report.iloc[0]
single_baseline_row = baseline_forward_report.iloc[0]
portfolio_delta = float(weighted_row["total_return"] - single_baseline_row["total_return"])
portfolio_score_delta = float(weighted_row["performance_score"] - single_baseline_row["performance_score"])
active_weights = strategy_weights[strategy_weights["active"]].copy()

param_lines = []
for row in provider_params.itertuples(index=False):
    param_lines.append(
        f"- `{row.provider}` selected `{int(row.fast_window)}/{int(row.slow_window)}` "
        f"with train return `{row.total_return:.2%}`, Sharpe `{row.train_sharpe:.2f}`, "
        f"max drawdown `{row.max_drawdown:.2%}`, and score `{row.performance_score:.4f}`."
    )

weight_lines = []
for row in active_weights.head(8).itertuples(index=False):
    weight_lines.append(
        f"- `{row.provider}` + `{row.forward_framework}` `{int(row.fast_window)}/{int(row.slow_window)}`: `{row.weight:.1%}`"
    )

analysis = f'''
### What Ran

- Loaded `{', '.join(PROVIDERS)}` from Quant Warehouse using `{PRICE_BASIS}`.
- Aligned every provider/symbol to `{DATE_ALIGNMENT}` before training or testing.
- Optimized `{len(FAST_WINDOWS) * len(SLOW_WINDOWS)}` SMA parameter combinations per provider with `{OPTIMIZATION_FRAMEWORK}` on `{WINDOW.train_start.date()}` through `{WINDOW.train_end.date()}`.
- Forward-tested the selected provider-specific parameters on `{', '.join(FORWARD_FRAMEWORKS)}` from `{WINDOW.test_start.date()}` through the latest common date.
- Built a filtered mean-variance strategy portfolio from candidates with train return >= `{MIN_TRAIN_RETURN:.2f}`, Sharpe >= `{MIN_TRAIN_SHARPE:.2f}`, and trades >= `{MIN_TRAIN_TRADES}`. The active portfolio used `{int(weighted_row.active_strategies)}` strategies across `{int(weighted_row.active_providers)}` providers and `{int(weighted_row.active_frameworks)}` frameworks.

### Selected Parameters

{chr(10).join(param_lines)}

### In-Sample Selected Benchmark

The single-strategy benchmark is selected without looking at out-of-sample results: choose the top in-sample provider/parameter row, then run the predeclared `{BASELINE_FORWARD_FRAMEWORK}` forward framework. That benchmark was `{baseline_forward.provider}` + `{baseline_forward.forward_framework}` using `{int(baseline_forward.fast_window)}/{int(baseline_forward.slow_window)}`. It returned `{baseline_forward.total_return:.2%}` with max drawdown `{baseline_forward.max_drawdown:.2%}` and performance score `{baseline_forward.performance_score:.4f}`.

The fastest single forward run was `{fastest_forward.provider}` + `{fastest_forward.forward_framework}` at `{fastest_forward.elapsed_seconds:.2f}` seconds. The slowest was `{slowest_forward.provider}` + `{slowest_forward.forward_framework}` at `{slowest_forward.elapsed_seconds:.2f}` seconds, so the fastest row was `{slowest_forward.elapsed_seconds / fastest_forward.elapsed_seconds:.2f}x` faster.

### Mean-Variance Strategy Portfolio

The filtered mean-variance portfolio returned `{weighted_row.total_return:.2%}` with max drawdown `{weighted_row.max_drawdown:.2%}` and performance score `{weighted_row.performance_score:.4f}`. Versus the in-sample selected single-strategy benchmark, the return delta was `{portfolio_delta:.2%}` and the performance-score delta was `{portfolio_score_delta:.4f}`.

Largest active weights:

{chr(10).join(weight_lines)}

### Vendor vs Framework Signal

For the visible selected-parameter forward rows, the descriptive variance split says `{perf_factor.dominant_factor}` explains more of the observed difference: provider share `{perf_factor.provider_share:.1%}`, framework share `{perf_factor.framework_share:.1%}`, residual share `{perf_factor.residual_share:.1%}`.

For total return, the dominant factor was `{return_factor.dominant_factor}` with provider share `{return_factor.provider_share:.1%}` and framework share `{return_factor.framework_share:.1%}`. For elapsed runtime, the dominant factor was `{speed_factor.dominant_factor}` with provider share `{speed_factor.provider_share:.1%}` and framework share `{speed_factor.framework_share:.1%}`.

### Design Takeaway

This is the orchestration pattern we wanted to test: one engine can be used for fast parameter search, independent forward-test jobs can validate selected artifacts on different event-driven engines, and a later portfolio job can consume those strategy artifacts without rewriting the data or strategy code. Quant Orchestrator should keep these as separate artifacts: provider-aligned data, optimizer result, selected parameters, framework-native forward results, normalized comparison tables, and portfolio-construction outputs.
'''

_ = pipeline_context.update({"optimization_top10": optimization_top10, "selected_parameters": selected_parameters, "forward_report": forward_report, "candidate_universe": candidate_universe, "strategy_weights": strategy_weights, "weighted_forward_report": weighted_forward_report, "factor_report": factor_report})
display(Markdown(analysis))


### What Ran

- Loaded `yfinance, fmp` from Quant Warehouse using `warehouse_splits_and_dividends_adjusted_ohlc`.
- Aligned every provider/symbol to `global_provider_symbol_intersection` before training or testing.
- Optimized `100` SMA parameter combinations per provider with `backtesting.py` on `2020-01-01` through `2025-12-31`.
- Forward-tested the selected provider-specific parameters on `zipline, nautilus` from `2026-01-01` through the latest common date.
- Built a filtered mean-variance strategy portfolio from candidates with train return >= `1.00`, Sharpe >= `0.50`, and trades >= `20`. The active portfolio used `9` strategies across `2` providers and `2` frameworks.

### Selected Parameters

- `yfinance` selected `35/60` with train return `180.23%`, Sharpe `1.29`, max drawdown `-28.82%`, and score `1.5141`.
- `fmp` selected `35/60` with train return `178.56%`, Sharpe `1.28`, max drawdown `-28.85%`, and score `1.4971`.

### In-Sample Selected Benchmark

The single-strategy benchmark is selected without looking at out-of-sample results: choose the top in-sample provider/parameter row, then run the predeclared `zipline` forward framework. That benchmark was `yfinance` + `zipline` using `35/60`. It returned `-2.44%` with max drawdown `-2.81%` and performance score `-0.0525`.

The fastest single forward run was `fmp` + `zipline` at `0.54` seconds. The slowest was `yfinance` + `nautilus` at `20.37` seconds, so the fastest row was `37.57x` faster.

### Mean-Variance Strategy Portfolio

The filtered mean-variance portfolio returned `-2.66%` with max drawdown `-3.01%` and performance score `-0.0567`. Versus the in-sample selected single-strategy benchmark, the return delta was `-0.22%` and the performance-score delta was `-0.0042`.

Largest active weights:

- `yfinance` + `nautilus` `35/60`: `34.6%`
- `yfinance` + `zipline` `35/60`: `34.0%`
- `fmp` + `zipline` `40/60`: `9.3%`
- `yfinance` + `nautilus` `30/60`: `7.9%`
- `fmp` + `zipline` `35/60`: `4.6%`
- `fmp` + `nautilus` `30/60`: `3.6%`
- `fmp` + `zipline` `30/60`: `2.4%`
- `yfinance` + `zipline` `40/60`: `2.3%`

### Vendor vs Framework Signal

For the visible selected-parameter forward rows, the descriptive variance split says `provider` explains more of the observed difference: provider share `55.9%`, framework share `0.5%`, residual share `43.6%`.

For total return, the dominant factor was `provider` with provider share `54.7%` and framework share `0.3%`. For elapsed runtime, the dominant factor was `framework` with provider share `0.0%` and framework share `100.0%`.

### Design Takeaway

This is the orchestration pattern we wanted to test: one engine can be used for fast parameter search, independent forward-test jobs can validate selected artifacts on different event-driven engines, and a later portfolio job can consume those strategy artifacts without rewriting the data or strategy code. Quant Orchestrator should keep these as separate artifacts: provider-aligned data, optimizer result, selected parameters, framework-native forward results, normalized comparison tables, and portfolio-construction outputs.
